# Notebook 24 — Route-Memory Policy Evaluation

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 23 used decompression forecasts to route before fallback occurs.

Notebook 24 evaluates routing policies across repeated trials:

- reactive routing,
- predictive routing,
- conservative predictive routing,
- aggressive predictive routing,
- CGCS-balanced routing.

Constraint view:
> route-memory policies should be compared by stability, fallback reduction, regret, and constraint alignment.

## Goals

1. Load Notebook 23 predictive constraint routing outputs when available.
2. Define multiple route-memory policies.
3. Evaluate policies across repeated stochastic trials.
4. Compare:
   - stability,
   - fallback rate,
   - reroute rate,
   - switch rate,
   - regret,
   - CGCS-style policy score.
5. Export CSV, JSON, Markdown report, and PNG figures.
6. Generate a Colab-downloadable output zip.

This notebook follows the Notebook 22/23 format:

- sectioned notebook,
- saved figures plus `plt.show()`,
- full linked Markdown report,
- `figures/` link style in report output.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 23 outputs

Uses:

```text
results/notebook23_predictive_constraint_routing.csv
```

If unavailable, this notebook creates a fallback synthetic routing stream.

In [ ]:
input_path = RESULTS_DIR / "notebook23_predictive_constraint_routing.csv"

if input_path.exists():
    base = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 23 output not found; creating fallback stream.")
    rng = np.random.default_rng(24)
    n = 240
    windows = np.arange(n)
    macro_routes = rng.choice(
        ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"],
        size=n,
        p=[0.34, 0.12, 0.20, 0.18, 0.16],
    )
    macro_cgcs_score = (
        0.55
        + 0.08 * np.sin(np.linspace(0, 8 * np.pi, n))
        + rng.normal(0, 0.08, n)
    )
    macro_cgcs_score[105:145] += 0.12
    macro_cgcs_score[35:55] -= 0.12
    macro_cgcs_score[205:230] -= 0.10
    macro_cgcs_score = np.clip(macro_cgcs_score, 0.20, 0.84)

    rolling_stability = pd.Series(macro_cgcs_score).rolling(12, min_periods=1).mean()
    rolling_volatility = pd.Series(macro_cgcs_score).rolling(10, min_periods=1).std().fillna(0)
    rolling_switch_rate = (
        pd.Series(macro_routes)
        .ne(pd.Series(macro_routes).shift())
        .rolling(15, min_periods=1)
        .mean()
    )
    rolling_residual = 1 - rolling_stability + rolling_volatility
    pressure_raw = (
        0.40 * rolling_switch_rate
        + 0.35 * rolling_volatility
        + 0.25 * rolling_residual
    )
    rolling_pressure = (pressure_raw - pressure_raw.min()) / (pressure_raw.max() - pressure_raw.min())
    decompression_event = ((macro_cgcs_score < 0.45) & (rolling_pressure > 0.55)).astype(int)
    future_decompression = (
        pd.Series(decompression_event)
        .rolling(5, min_periods=1)
        .max()
        .shift(-5)
        .fillna(0)
        .astype(int)
    )
    probability = np.clip(
        0.15
        + 0.55 * rolling_pressure
        + 0.25 * (1 - macro_cgcs_score)
        + rng.normal(0, 0.08, n),
        0,
        1,
    )
    base = pd.DataFrame({
        "window_id": windows,
        "macro_route": macro_routes,
        "macro_cgcs_score": macro_cgcs_score,
        "rolling_stability": rolling_stability,
        "rolling_volatility": rolling_volatility,
        "rolling_switch_rate": rolling_switch_rate,
        "rolling_residual": rolling_residual,
        "rolling_pressure": rolling_pressure,
        "decompression_event": decompression_event,
        "future_decompression": future_decompression,
        "decompression_probability": probability,
    })

base = base.copy().reset_index(drop=True)

if "window_id" not in base.columns:
    if "window" in base.columns:
        base["window_id"] = base["window"]
    else:
        base["window_id"] = np.arange(len(base))

if "macro_route" not in base.columns:
    base["macro_route"] = "macro_unknown"

for c in [
    "macro_cgcs_score",
    "rolling_stability",
    "rolling_volatility",
    "rolling_switch_rate",
    "rolling_residual",
    "rolling_pressure",
    "decompression_probability",
]:
    if c not in base.columns:
        base[c] = 0.5
    base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0.5).clip(0, 1)

if "decompression_event" not in base.columns:
    base["decompression_event"] = ((base["macro_cgcs_score"] < 0.45) & (base["rolling_pressure"] > 0.55)).astype(int)

if "future_decompression" not in base.columns:
    base["future_decompression"] = (
        pd.Series(base["decompression_event"])
        .rolling(5, min_periods=1)
        .max()
        .shift(-5)
        .fillna(0)
        .astype(int)
    )

base.head()

## Define route-memory policies

Each policy receives the same stream and returns:

- gate state,
- routing action,
- stability estimate,
- switching estimate,
- fallback/reroute counts.

In [ ]:
policies = {
    "reactive": {
        "reroute_threshold": 1.01,
        "fallback_threshold": 1.01,
        "fallback_on_event": True,
        "accepted_threshold": 0.70,
        "reroute_bonus": 0.00,
        "fallback_penalty": 0.22,
        "future_penalty": 0.10,
    },
    "predictive": {
        "reroute_threshold": 0.50,
        "fallback_threshold": 0.70,
        "fallback_on_event": False,
        "accepted_threshold": 0.70,
        "reroute_bonus": 0.12,
        "fallback_penalty": 0.11,
        "future_penalty": 0.04,
    },
    "conservative_predictive": {
        "reroute_threshold": 0.62,
        "fallback_threshold": 0.82,
        "fallback_on_event": False,
        "accepted_threshold": 0.72,
        "reroute_bonus": 0.09,
        "fallback_penalty": 0.09,
        "future_penalty": 0.05,
    },
    "aggressive_predictive": {
        "reroute_threshold": 0.38,
        "fallback_threshold": 0.62,
        "fallback_on_event": False,
        "accepted_threshold": 0.68,
        "reroute_bonus": 0.10,
        "fallback_penalty": 0.13,
        "future_penalty": 0.03,
    },
    "cgcs_balanced": {
        "reroute_threshold": 0.52,
        "fallback_threshold": 0.74,
        "fallback_on_event": False,
        "accepted_threshold": 0.70,
        "reroute_bonus": 0.10,
        "fallback_penalty": 0.10,
        "future_penalty": 0.04,
    },
}

def evaluate_policy(df, name, cfg, trial_seed=0):
    rng = np.random.default_rng(trial_seed)
    rows = []

    for _, row in df.iterrows():
        risk = float(row["decompression_probability"])
        pressure = float(row["rolling_pressure"])
        cgcs = float(row["macro_cgcs_score"])
        event = int(row["decompression_event"])
        future = int(row["future_decompression"])
        base_stability = float(row["rolling_stability"])

        noisy_risk = np.clip(risk + rng.normal(0, 0.025), 0, 1)
        noisy_pressure = np.clip(pressure + rng.normal(0, 0.025), 0, 1)

        if cfg["fallback_on_event"] and event == 1:
            gate = "fallback"
            action = "reactive_fallback"
        elif noisy_risk >= cfg["fallback_threshold"] and noisy_pressure >= 0.55:
            gate = "fallback"
            action = "protective_fallback"
        elif noisy_risk >= cfg["reroute_threshold"]:
            gate = "reroute"
            action = "predictive_reroute"
        elif cgcs >= cfg["accepted_threshold"]:
            gate = "accepted"
            action = "retain_compressed"
        else:
            gate = "watch"
            action = "monitor"

        stability = np.clip(
            base_stability
            + cfg["reroute_bonus"] * (gate == "reroute")
            + 0.06 * (gate == "accepted")
            - cfg["fallback_penalty"] * (gate == "fallback")
            - cfg["future_penalty"] * future,
            0,
            1,
        )

        constraint_score = np.clip(
            0.40 * stability
            + 0.25 * (1 - noisy_pressure)
            + 0.20 * cgcs
            + 0.15 * (1 - int(gate == "fallback")),
            0,
            1,
        )

        cost = (
            1.00 * (gate == "fallback")
            + 0.35 * (gate == "reroute")
            + 0.15 * (gate == "watch")
            + 0.10 * future
            + 0.20 * noisy_pressure
            - 0.30 * stability
        )

        rows.append({
            "window_id": int(row["window_id"]),
            "macro_route": row["macro_route"],
            "policy": name,
            "gate": gate,
            "action": action,
            "stability": stability,
            "constraint_score": constraint_score,
            "policy_cost": cost,
            "risk": noisy_risk,
            "pressure": noisy_pressure,
            "future_decompression": future,
            "decompression_event": event,
        })

    out = pd.DataFrame(rows)
    out["switch"] = out["gate"].ne(out["gate"].shift()).astype(int)
    out["switch_rate"] = out["switch"].rolling(15, min_periods=1).mean()
    return out

# Preview one policy.
evaluate_policy(base, "predictive", policies["predictive"], trial_seed=240).head()

## Run repeated trials

Repeated trials add small risk/pressure perturbations so policy stability can be compared as a distribution rather than a single trace.

In [ ]:
n_trials = 40

all_trials = []
for trial in range(n_trials):
    for policy_name, cfg in policies.items():
        trial_df = evaluate_policy(base, policy_name, cfg, trial_seed=10_000 + trial)
        trial_df["trial"] = trial
        all_trials.append(trial_df)

eval_df = pd.concat(all_trials, ignore_index=True)

eval_df.head()

## Policy summaries

Regret is computed relative to the best policy cost observed for each `(trial, window)` pair.

In [ ]:
best_cost = (
    eval_df.groupby(["trial", "window_id"])["policy_cost"]
    .min()
    .rename("best_policy_cost")
    .reset_index()
)

eval_df = eval_df.merge(best_cost, on=["trial", "window_id"], how="left")
eval_df["regret"] = eval_df["policy_cost"] - eval_df["best_policy_cost"]

policy_summary = (
    eval_df.groupby("policy")
    .agg(
        windows=("window_id", "count"),
        mean_stability=("stability", "mean"),
        std_stability=("stability", "std"),
        mean_constraint_score=("constraint_score", "mean"),
        mean_cost=("policy_cost", "mean"),
        mean_regret=("regret", "mean"),
        fallback_rate=("gate", lambda s: (s == "fallback").mean()),
        reroute_rate=("gate", lambda s: (s == "reroute").mean()),
        watch_rate=("gate", lambda s: (s == "watch").mean()),
        accepted_rate=("gate", lambda s: (s == "accepted").mean()),
        mean_switch_rate=("switch_rate", "mean"),
    )
    .reset_index()
    .sort_values("mean_constraint_score", ascending=False)
)

trial_summary = (
    eval_df.groupby(["trial", "policy"])
    .agg(
        mean_stability=("stability", "mean"),
        mean_constraint_score=("constraint_score", "mean"),
        mean_cost=("policy_cost", "mean"),
        mean_regret=("regret", "mean"),
        fallback_rate=("gate", lambda s: (s == "fallback").mean()),
        reroute_rate=("gate", lambda s: (s == "reroute").mean()),
        mean_switch_rate=("switch_rate", "mean"),
    )
    .reset_index()
)

route_policy_summary = (
    eval_df.groupby(["macro_route", "policy"])
    .agg(
        windows=("window_id", "count"),
        mean_stability=("stability", "mean"),
        mean_constraint_score=("constraint_score", "mean"),
        mean_regret=("regret", "mean"),
        fallback_rate=("gate", lambda s: (s == "fallback").mean()),
        reroute_rate=("gate", lambda s: (s == "reroute").mean()),
    )
    .reset_index()
)

policy_summary

## Transition matrices

Build one gate transition matrix per policy from the first trial for readable reporting.

In [ ]:
gate_order = ["accepted", "watch", "reroute", "fallback"]

def transition_matrix(labels, order):
    mat = pd.DataFrame(0.0, index=order, columns=order)
    labels = list(labels)
    for i in range(len(labels) - 1):
        mat.loc[labels[i], labels[i + 1]] += 1
    return mat.div(mat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

transition_tables = {}
for policy_name in policies:
    part = eval_df[(eval_df["policy"] == policy_name) & (eval_df["trial"] == 0)]
    transition_tables[policy_name] = transition_matrix(part["gate"], gate_order)

transition_tables["predictive"]

## Export result artifacts

In [ ]:
eval_csv = RESULTS_DIR / "notebook24_route_memory_policy_evaluation.csv"
eval_json = RESULTS_DIR / "notebook24_route_memory_policy_evaluation.json"
policy_summary_csv = RESULTS_DIR / "notebook24_policy_summary.csv"
trial_summary_csv = RESULTS_DIR / "notebook24_trial_summary.csv"
route_policy_summary_csv = RESULTS_DIR / "notebook24_route_policy_summary.csv"

eval_df.to_csv(eval_csv, index=False)
eval_df.to_json(eval_json, orient="records", indent=2)
policy_summary.to_csv(policy_summary_csv, index=False)
trial_summary.to_csv(trial_summary_csv, index=False)
route_policy_summary.to_csv(route_policy_summary_csv, index=False)

transition_paths = {}
for policy_name, mat in transition_tables.items():
    p = RESULTS_DIR / f"notebook24_{policy_name}_gate_transition_matrix.csv"
    mat.to_csv(p)
    transition_paths[policy_name] = p

print("Saved:", eval_csv)
print("Saved:", eval_json)
print("Saved:", policy_summary_csv)
print("Saved:", trial_summary_csv)
print("Saved:", route_policy_summary_csv)
for k, v in transition_paths.items():
    print("Saved:", k, v)

## Figure 1 — Policy constraint score

In [ ]:
policy_score_fig = FIGURES_DIR / "notebook24_policy_constraint_score.png"

plot_df = policy_summary.sort_values("mean_constraint_score")

plt.figure(figsize=(10, 6))
plt.bar(plot_df["policy"], plot_df["mean_constraint_score"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Mean constraint score")
plt.title("Route-Memory Policy Evaluation: Mean Constraint Score")
plt.tight_layout()
plt.savefig(policy_score_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", policy_score_fig)

## Figure 2 — Policy regret

In [ ]:
regret_fig = FIGURES_DIR / "notebook24_policy_regret.png"

plot_df = policy_summary.sort_values("mean_regret")

plt.figure(figsize=(10, 6))
plt.bar(plot_df["policy"], plot_df["mean_regret"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Mean regret")
plt.title("Route-Memory Policy Evaluation: Mean Regret")
plt.tight_layout()
plt.savefig(regret_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", regret_fig)

## Figure 3 — Fallback vs reroute rates

In [ ]:
fallback_reroute_fig = FIGURES_DIR / "notebook24_fallback_vs_reroute_rates.png"

x = np.arange(len(policy_summary))
width = 0.35
plot_df = policy_summary.sort_values("policy")

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, plot_df["fallback_rate"], width, label="fallback rate")
plt.bar(x + width/2, plot_df["reroute_rate"], width, label="reroute rate")
plt.xticks(x, plot_df["policy"], rotation=35, ha="right")
plt.ylabel("Rate")
plt.title("Route-Memory Policy Evaluation: Fallback vs Reroute Rates")
plt.legend()
plt.tight_layout()
plt.savefig(fallback_reroute_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", fallback_reroute_fig)

## Figure 4 — Stability distribution

In [ ]:
stability_distribution_fig = FIGURES_DIR / "notebook24_stability_distribution.png"

plt.figure(figsize=(10, 6))
data = [trial_summary[trial_summary["policy"] == p]["mean_stability"].values for p in sorted(policies)]
plt.boxplot(data, labels=sorted(policies))
plt.xticks(rotation=35, ha="right")
plt.ylabel("Trial mean stability")
plt.title("Route-Memory Policy Evaluation: Stability Distribution")
plt.tight_layout()
plt.savefig(stability_distribution_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", stability_distribution_fig)

## Figure 5 — Switch-rate distribution

In [ ]:
switch_distribution_fig = FIGURES_DIR / "notebook24_switch_rate_distribution.png"

plt.figure(figsize=(10, 6))
data = [trial_summary[trial_summary["policy"] == p]["mean_switch_rate"].values for p in sorted(policies)]
plt.boxplot(data, labels=sorted(policies))
plt.xticks(rotation=35, ha="right")
plt.ylabel("Trial mean switch rate")
plt.title("Route-Memory Policy Evaluation: Switch-Rate Distribution")
plt.tight_layout()
plt.savefig(switch_distribution_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", switch_distribution_fig)

## Figure 6 — Route-policy heatmap

In [ ]:
route_policy_heatmap_fig = FIGURES_DIR / "notebook24_route_policy_constraint_heatmap.png"

heat = route_policy_summary.pivot(
    index="macro_route",
    columns="policy",
    values="mean_constraint_score",
).fillna(0)

plt.figure(figsize=(10, 6))
plt.imshow(heat.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(heat.columns)), heat.columns, rotation=35, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.colorbar(label="Mean constraint score")
plt.title("Route-Memory Policy Evaluation: Route × Policy Constraint Score")
plt.tight_layout()
plt.savefig(route_policy_heatmap_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", route_policy_heatmap_fig)

## Figure 7 — Trial constraint score timeline

In [ ]:
trial_score_fig = FIGURES_DIR / "notebook24_trial_constraint_score_timeline.png"

plt.figure(figsize=(14, 6))
for policy_name in sorted(policies):
    part = trial_summary[trial_summary["policy"] == policy_name].sort_values("trial")
    plt.plot(part["trial"], part["mean_constraint_score"], label=policy_name)
plt.xlabel("Trial")
plt.ylabel("Mean constraint score")
plt.title("Route-Memory Policy Evaluation: Trial Constraint Score")
plt.legend()
plt.tight_layout()
plt.savefig(trial_score_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", trial_score_fig)

## Figure 8 — Predictive gate transition matrix

In [ ]:
predictive_transition_fig = FIGURES_DIR / "notebook24_predictive_gate_transition_matrix.png"

mat = transition_tables["predictive"]

plt.figure(figsize=(7, 6))
plt.imshow(mat.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(mat.columns)), mat.columns, rotation=35, ha="right")
plt.yticks(range(len(mat.index)), mat.index)
plt.colorbar(label="Transition probability")
plt.title("Route-Memory Policy Evaluation: Predictive Gate Transition Matrix")
plt.xlabel("Next gate")
plt.ylabel("Current gate")
plt.tight_layout()
plt.savefig(predictive_transition_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", predictive_transition_fig)

## Figure 9 — Policy recommendation summary

In [ ]:
recommendation_fig = FIGURES_DIR / "notebook24_policy_recommendation_summary.png"

recommendation = policy_summary.copy()
recommendation["rank_score"] = (
    0.40 * recommendation["mean_constraint_score"]
    + 0.25 * recommendation["mean_stability"]
    + 0.20 * (1 - recommendation["mean_regret"] / max(recommendation["mean_regret"].max(), 1e-9))
    + 0.15 * (1 - recommendation["fallback_rate"])
)
recommendation = recommendation.sort_values("rank_score")

plt.figure(figsize=(10, 6))
plt.bar(recommendation["policy"], recommendation["rank_score"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Recommendation score")
plt.title("Route-Memory Policy Evaluation: Policy Recommendation Score")
plt.tight_layout()
plt.savefig(recommendation_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", recommendation_fig)

## Full Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_24_route_memory_policy_evaluation.md"

# Repo-relative report links.
eval_csv_link = "results/notebook24_route_memory_policy_evaluation.csv"
eval_json_link = "results/notebook24_route_memory_policy_evaluation.json"
policy_summary_csv_link = "results/notebook24_policy_summary.csv"
trial_summary_csv_link = "results/notebook24_trial_summary.csv"
route_policy_summary_csv_link = "results/notebook24_route_policy_summary.csv"

transition_links = {
    policy_name: f"results/notebook24_{policy_name}_gate_transition_matrix.csv"
    for policy_name in sorted(policies)
}

policy_score_link = "figures/notebook24_policy_constraint_score.png"
regret_link = "figures/notebook24_policy_regret.png"
fallback_reroute_link = "figures/notebook24_fallback_vs_reroute_rates.png"
stability_distribution_link = "figures/notebook24_stability_distribution.png"
switch_distribution_link = "figures/notebook24_switch_rate_distribution.png"
route_policy_heatmap_link = "figures/notebook24_route_policy_constraint_heatmap.png"
trial_score_link = "figures/notebook24_trial_constraint_score_timeline.png"
predictive_transition_link = "figures/notebook24_predictive_gate_transition_matrix.png"
recommendation_link = "figures/notebook24_policy_recommendation_summary.png"

best_policy = recommendation.sort_values("rank_score", ascending=False).iloc[0]["policy"]

report_lines = [
    "# Report 24 — Route-Memory Policy Evaluation",
    "",
    "This report compares route-memory policies across repeated trials.",
    "",
    "Constraint view:",
    "> route-memory policies should be compared by stability, fallback reduction, regret, and constraint alignment.",
    "",
    "## Generated outputs",
    "",
    f'- Policy evaluation CSV: <a href="{eval_csv_link}">`{eval_csv_link}`</a>',
    f'- Policy evaluation JSON: <a href="{eval_json_link}">`{eval_json_link}`</a>',
    f'- Policy summary CSV: <a href="{policy_summary_csv_link}">`{policy_summary_csv_link}`</a>',
    f'- Trial summary CSV: <a href="{trial_summary_csv_link}">`{trial_summary_csv_link}`</a>',
    f'- Route-policy summary CSV: <a href="{route_policy_summary_csv_link}">`{route_policy_summary_csv_link}`</a>',
]

for policy_name, link in transition_links.items():
    report_lines.append(
        f'- {policy_name} gate transition matrix CSV: <a href="{link}">`{link}`</a>'
    )

report_lines += [
    f'- Figure: <a href="{policy_score_link}">`{policy_score_link}`</a>',
    f'- Figure: <a href="{regret_link}">`{regret_link}`</a>',
    f'- Figure: <a href="{fallback_reroute_link}">`{fallback_reroute_link}`</a>',
    f'- Figure: <a href="{stability_distribution_link}">`{stability_distribution_link}`</a>',
    f'- Figure: <a href="{switch_distribution_link}">`{switch_distribution_link}`</a>',
    f'- Figure: <a href="{route_policy_heatmap_link}">`{route_policy_heatmap_link}`</a>',
    f'- Figure: <a href="{trial_score_link}">`{trial_score_link}`</a>',
    f'- Figure: <a href="{predictive_transition_link}">`{predictive_transition_link}`</a>',
    f'- Figure: <a href="{recommendation_link}">`{recommendation_link}`</a>',
    "",
    "## Summary",
    "",
    policy_summary.to_markdown(index=False),
    "",
    "## Trial summary preview",
    "",
    trial_summary.head(15).to_markdown(index=False),
    "",
    "## Route-policy summary",
    "",
    route_policy_summary.to_markdown(index=False),
    "",
    "## Policy recommendation",
    "",
    recommendation.sort_values("rank_score", ascending=False).to_markdown(index=False),
    "",
    "## Predictive gate transition probabilities",
    "",
    transition_tables["predictive"].to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Reactive routing minimizes early intervention but waits until decompression is already observed.",
    "- Predictive routing can reduce reactive fallback handling by inserting reroute states before collapse.",
    "- Conservative predictive routing reduces unnecessary reroutes but may miss some avoidable fallback windows.",
    "- Aggressive predictive routing increases reroute pressure and may raise switching cost.",
    "- CGCS-balanced routing weighs stability, pressure, fallback avoidance, and route switching together.",
    f"- Best policy by recommendation score in this run: `{best_policy}`.",
    "",
    "## Next step",
    "",
    "Notebook 25 can build policy-regret phase diagrams:",
    "- vary forecast threshold,",
    "- vary fallback penalty,",
    "- vary reroute cost,",
    "- map stable regions of route-memory policy choice.",
]

report_path.write_text("\n".join(report_lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if running in Google Colab.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook24_route_memory_policy_evaluation_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook24_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_24_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))